# Exploratory Data Analysis

Overview of the UCI Student Performance (Math) dataset: distributions, correlations, and key relationships with the target variable.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data import load_data

sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. Dataset Overview

In [ ]:
df = load_data()
print(f"Shape: {df.shape}")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print("Missing values per column:")
print(df.isnull().sum()[df.isnull().sum() > 0])
if df.isnull().sum().sum() == 0:
    print("No missing values.")

## 2. Target Distribution

In [ ]:
df["at_risk"] = (df["G3"] < 10).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# G3 distribution
axes[0].hist(df["G3"], bins=20, edgecolor="black", alpha=0.7)
axes[0].axvline(x=10, color="red", linestyle="--", label="at_risk threshold (10)")
axes[0].set_xlabel("Final Grade (G3)")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of Final Grade")
axes[0].legend()

# at_risk class balance
counts = df["at_risk"].value_counts()
axes[1].bar(["Not at risk (0)", "At risk (1)"], counts.values, color=["steelblue", "salmon"], edgecolor="black")
axes[1].set_ylabel("Count")
axes[1].set_title(f"Class Balance — at_risk (1): {counts[1]}, not (0): {counts[0]}")

plt.tight_layout()
plt.show()

## 3. Numeric Feature Distributions

In [ ]:
num_cols = df.select_dtypes(exclude="object").columns.drop(["G3", "at_risk"])

fig, axes = plt.subplots(4, 4, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df[col], bins=15, edgecolor="black", alpha=0.7)
    axes[i].set_title(col)

# Hide unused subplots
for j in range(len(num_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Numeric Feature Distributions", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 4. Categorical Feature Distributions

In [ ]:
cat_cols = df.select_dtypes(include="object").columns

fig, axes = plt.subplots(3, 3, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    sns.countplot(data=df, x=col, ax=axes[i], order=df[col].value_counts().index)
    axes[i].set_title(col)
    axes[i].tick_params(axis="x", rotation=45)

for j in range(len(cat_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Categorical Feature Distributions", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 5. Correlation Heatmap

In [ ]:
corr = df.select_dtypes(exclude="object").corr()

plt.figure(figsize=(14, 10))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, linewidths=0.5)
plt.title("Correlation Heatmap (Numeric Features)")
plt.tight_layout()
plt.show()

## 6. Key Bivariate Relationships with at_risk

In [ ]:
key_features = ["failures", "studytime", "absences", "Medu", "Fedu", "goout", "Dalc", "Walc"]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(key_features):
    sns.boxplot(data=df, x="at_risk", y=col, ax=axes[i], palette=["steelblue", "salmon"])
    axes[i].set_xticklabels(["Not at risk", "At risk"])
    axes[i].set_title(f"{col} vs at_risk")

plt.suptitle("Key Features by At-Risk Status", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for i, col in enumerate(["higher", "schoolsup", "romantic"]):
    ct = pd.crosstab(df[col], df["at_risk"], normalize="index")
    ct.plot(kind="bar", stacked=True, ax=axes[i], color=["steelblue", "salmon"])
    axes[i].set_title(f"{col} vs at_risk (proportion)")
    axes[i].set_ylabel("Proportion")
    axes[i].legend(["Not at risk", "At risk"])
    axes[i].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()